# 07. 모델 해석

`06_model_finalization`에서 확정해 저장한 최종 모델과 변수 조합을 기준으로 Permutation Importance를 계산한다.

이 노트북에서는 Permutation Importance를 측정하여 각 변수의 실질적인 예측 기여도를 확인하고, 트리 기반 Feature Importance와 비교한다.

- Permutation Importance는 모델 해석용 보조 지표이며, 인과 효과로 해석하지 않는다.
- 테스트 세트 결과는 변수 기여도 해석에만 사용하며, 추가 모델 선택이나 변경을 수행하지 않는다.



## 1. 라이브러리 및 경로 설정


In [ ]:
import ast
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / '.matplotlib'))
os.environ.setdefault('XDG_CACHE_HOME', str(PROJECT_ROOT / '.cache'))
os.environ.setdefault('LOKY_MAX_CPU_COUNT', str(os.cpu_count() or 1))
(PROJECT_ROOT / '.matplotlib').mkdir(exist_ok=True)
(PROJECT_ROOT / '.cache').mkdir(exist_ok=True)
(PROJECT_ROOT / 'reports').mkdir(exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

FEATURE_DATA_PATH = PROJECT_ROOT / 'data/processed/seoul_apt_trade_2025_features.csv'
MODELING_DATA_PATH = PROJECT_ROOT / 'data/processed/modeling_dataset.csv'

FINAL_SCORE_PATH = PROJECT_ROOT / 'reports/06_final_model_scores.csv'
PERM_IMPORTANCE_PATH = PROJECT_ROOT / 'reports/06_2_permutation_importance.csv'
FINAL_FIGURE_DIR = PROJECT_ROOT / 'reports/figures'
FINAL_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 2. 데이터 로드

06 단계와 동일하다.


In [ ]:
if FEATURE_DATA_PATH.exists():
    source_df = pd.read_csv(FEATURE_DATA_PATH, encoding='utf-8-sig')
    source_name = FEATURE_DATA_PATH.name
elif MODELING_DATA_PATH.exists():
    source_df = pd.read_csv(MODELING_DATA_PATH, encoding='utf-8-sig')
    source_name = MODELING_DATA_PATH.name
else:
    raise FileNotFoundError('모델 검증에 필요한 CSV가 없습니다. 03 또는 05 노트북을 먼저 실행하세요.')

source_df['contract_date'] = pd.to_datetime(source_df['contract_date'])
source_df = source_df.sort_values('contract_date').reset_index(drop=True)

print(f'loaded: {source_name}')
source_df.shape


## 3. 최종 모델 설정

06 단계에서 확정한 변수 조합과 하이퍼파라미터를 그대로 사용한다.


In [ ]:
target_col = 'price_10k_krw'

# 06에서 저장한 최종 모델 정보 로드
if not FINAL_SCORE_PATH.exists():
    raise FileNotFoundError(f'06_final_model_scores.csv 없음. 06 노트북을 먼저 실행하세요: {FINAL_SCORE_PATH}')

final_score_row = pd.read_csv(FINAL_SCORE_PATH, encoding='utf-8-sig').iloc[0]
best_baseline_model = final_score_row['model']
best_feature_set = final_score_row['feature_set']
best_params = ast.literal_eval(final_score_row['params'])

print(f'최종 모델    : {best_baseline_model}')
print(f'변수 조합    : {best_feature_set}')
print(f'하이퍼파라미터: {best_params}')

# 06의 feature_sets_location과 동일한 변수 구성
feature_set_configs = {
    'reduced_time_age': {
        'numeric': [
            'area_m2', 'floor', 'age', 'contract_month',
            'distance_to_cbd_km', 'distance_to_ybd_km', 'distance_to_gbd_km',
            'nearest_subway_distance_km', 'nearest_hospital_distance_km',
            'large_mart_count_within_1km',
        ],
        'categorical': ['gu', 'law_dong'],
    },
    'nearest_business_district': {
        'numeric': [
            'area_m2', 'floor', 'age', 'contract_month',
            'nearest_business_district_distance_km',
            'nearest_subway_distance_km', 'nearest_hospital_distance_km',
            'large_mart_count_within_1km',
        ],
        'categorical': ['gu', 'law_dong', 'nearest_business_district'],
    },
    'no_distance': {
        'numeric': [
            'area_m2', 'floor', 'age', 'contract_month',
            'large_mart_count_within_1km',
        ],
        'categorical': ['gu', 'law_dong'],
    },
    'no_region': {
        'numeric': [
            'area_m2', 'floor', 'age', 'contract_month',
            'distance_to_cbd_km', 'distance_to_ybd_km', 'distance_to_gbd_km',
            'nearest_subway_distance_km', 'nearest_hospital_distance_km',
            'large_mart_count_within_1km',
        ],
        'categorical': [],
    },
    'no_region_nearest_bd': {
        'numeric': [
            'area_m2', 'floor', 'age', 'contract_month',
            'nearest_business_district_distance_km',
            'nearest_subway_distance_km', 'nearest_hospital_distance_km',
            'large_mart_count_within_1km',
        ],
        'categorical': ['nearest_business_district'],
    },
}

if best_feature_set not in feature_set_configs:
    raise KeyError(f'알 수 없는 feature_set: {best_feature_set}')

best_config = feature_set_configs[best_feature_set]
numeric_features = best_config['numeric']
categorical_features = best_config['categorical']

columns = [target_col, 'contract_date', *numeric_features, *categorical_features]
df = source_df[columns].dropna().sort_values('contract_date').reset_index(drop=True)

train_end = pd.Timestamp('2025-09-01')
valid_end = pd.Timestamp('2025-11-01')
train_df = df[df['contract_date'] < train_end].copy()
valid_df = df[(df['contract_date'] >= train_end) & (df['contract_date'] < valid_end)].copy()
test_df = df[df['contract_date'] >= valid_end].copy()

X_train = train_df[numeric_features + categorical_features]
y_train = train_df[target_col]
X_valid = valid_df[numeric_features + categorical_features]
y_valid = valid_df[target_col]
X_test = test_df[numeric_features + categorical_features]
y_test = test_df[target_col]

print(f'train: {len(train_df)}, valid: {len(valid_df)}, test: {len(test_df)}')

## 4. 최종 모델 학습

Permutation Importance는 학습된 모델에 평가 세트를 넣어 측정한다.
06 단계와 동일하게 train+valid로 재학습한 뒤 테스트 세트에서 측정한다.


In [ ]:
model_classes = {
    'random_forest': RandomForestRegressor,
    'extra_trees': ExtraTreesRegressor,
}

if best_baseline_model not in model_classes:
    raise KeyError(f'알 수 없는 model: {best_baseline_model}')

final_estimator = model_classes[best_baseline_model](
    random_state=RANDOM_STATE,
    n_jobs=-1,
    **best_params,
)

final_model = Pipeline([
    ('preprocess', ColumnTransformer(
        transformers=[
            ('num', SimpleImputer(strategy='median'), numeric_features),
            ('cat', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
            ]), categorical_features),
        ],
        remainder='drop',
    )),
    ('model', final_estimator),
])

X_train_final = pd.concat([X_train, X_valid])
y_train_final = pd.concat([y_train, y_valid])

final_model.fit(X_train_final, y_train_final)

test_pred = final_model.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
print(f'test RMSE: {test_rmse:,.0f} 만원')


## 5. Permutation Importance

각 변수를 무작위로 섞었을 때 테스트 RMSE가 얼마나 증가하는지를 측정한다.
트리 기반 feature importance와 달리 변수 간 상관관계에 덜 영향받는다.
`n_repeats=10`으로 측정 안정성을 확보한다.


In [ ]:
perm_result = permutation_importance(
    final_model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=1,
    scoring='neg_root_mean_squared_error',
)


In [ ]:
all_features = numeric_features + categorical_features
perm_df = pd.DataFrame({
    'feature': all_features,
    'importance_mean': perm_result.importances_mean,
    'importance_std': perm_result.importances_std,
}).sort_values('importance_mean', ascending=False).reset_index(drop=True)

perm_df.to_csv(PERM_IMPORTANCE_PATH, index=False, encoding='utf-8-sig')
print(f'permutation importance saved: {PERM_IMPORTANCE_PATH}')
perm_df


In [ ]:
# 시각화
# 막대는 평균 중요도, 끝의 선은 반복 permutation 결과의 표준편차
plot_df = perm_df.sort_values('importance_mean')
fig, ax = plt.subplots(figsize=(8, max(4, len(plot_df) * 0.35)))

ax.barh(plot_df['feature'], plot_df['importance_mean'] / 10_000, xerr=plot_df['importance_std'] / 10_000, color='#4C78A8', alpha=0.85)
ax.axvline(0, color='red', linewidth=1, linestyle='--')
ax.set_xlabel('Test RMSE Increase (100M KRW)')
ax.set_title('Permutation Importance — Final Test Set')
ax.grid(axis='x', alpha=0.25)

fig.tight_layout()
display(fig)
fig.savefig(FINAL_FIGURE_DIR / '06_2_permutation_importance.png', dpi=150, bbox_inches='tight')
plt.close(fig)


## 6. 결과 해석 및 정리

### Permutation Importance vs Feature Importance 비교

Permutation Importance 결과를 통해 트리 기반 Feature Importance에서 높게 나타난 변수들이 실제 예측 성능에도 얼마나 기여하는지 확인한다. 두 지표의 상위 변수가 유사하게 나타나면 변수 중요도 편중이 단순한 트리 분기 특성 때문이 아니라 실제 예측 기여도와도 관련된 것으로 해석할 수 있다.

### 변수 중요도 편중에 대한 판단

상위 변수와 RMSE 증가폭은 위 표와 그래프의 최신 실행 결과를 기준으로 해석한다. 중요도가 높은 변수는 해당 변수를 무작위로 섞었을 때 테스트 RMSE가 크게 증가했다는 뜻이므로, 최종 모델의 예측에 상대적으로 큰 영향을 준 변수로 볼 수 있다.

Permutation Importance는 최종 모델을 다시 선택하기 위한 기준이 아니라, 06 단계에서 확정한 모델의 예측 구조를 설명하기 위한 보조 지표로 사용한다.
